# GP07 — Measurements: Ενδεικτική λύση

Οι μετρήσεις γίνονται μετά από reprojection σε EPSG:2100. `area` επιστρέφει m², `length` m και οι νέες columns μετατρέπουν τις τιμές σε hectares/km. Οι assertions ελέγχουν CRS, geometry, θετικά μεγέθη και τάξη μεγέθους.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
root = Path('data/gp07')
areas = gpd.read_file(root / 'measurement_areas.geojson')
routes = gpd.read_file(root / 'measurement_routes.geojson')
sites_df = pd.read_csv(root / 'measurement_sites.csv')
sites = gpd.GeoDataFrame(sites_df, geometry=gpd.points_from_xy(sites_df.longitude, sites_df.latitude), crs='EPSG:4326')
target_crs = 'EPSG:2100'
areas_m, routes_m, sites_m = (x.to_crs(target_crs) for x in (areas, routes, sites))


In [ ]:
areas_m['area_m2'] = areas_m.geometry.area
areas_m['area_ha'] = areas_m['area_m2'] / 10_000
routes_m['length_m'] = routes_m.geometry.length
routes_m['length_km'] = routes_m['length_m'] / 1_000
distance_m = sites_m.geometry.iloc[0].distance(sites_m.geometry.iloc[1])
print(areas_m[['area_id','area_m2','area_ha']])
print(routes_m[['route_id','length_m','length_km']])
print(f'P01–P02: {distance_m:,.0f} m ({distance_m/1000:.2f} km)')


In [ ]:
for frame, expected, column in [(areas_m, 2, 'area_m2'), (routes_m, 2, 'length_m')]:
    assert len(frame) == expected
    assert frame.crs.to_epsg() == 2100
    assert frame.geometry.notna().all()
    assert (frame[column] > 0).all()
    print('bounds:', frame.total_bounds)
assert 3_000 < distance_m < 10_000
assert areas_m.geom_type.eq('Polygon').all()
assert routes_m.geom_type.eq('LineString').all()


In [ ]:
areas_m.to_file('gp07_areas_measured.gpkg', layer='areas', driver='GPKG')
routes_m.to_file('gp07_routes_measured.gpkg', layer='routes', driver='GPKG')
print('exported measured layers')
